# Combining tables: concatenation vs joins

_Data Wrangling_

**Stack rows with concat, line up rows on a key with merge — then check the count before and after.**

Think of two spreadsheets. Concatenation physically stacks them &mdash; either rows on
       top of rows (you need the same columns) or columns beside columns (you need the rows in the
       same order). It does no matching; it just glues.

       A join is smarter. You name a key column &mdash; say cust_id &mdash;
       and the join walks one table's keys, finds the matching key in the other table, and pastes the two
       rows together. The whole behavior of a join is decided by one question: what do you do with keys
       that appear in one table but not the other? That single question gives the four join types.

---

This notebook is a practice scaffold for the **Combining tables: concatenation vs joins** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas

In [ ]:
import pandas as pd

# === Two small tables ===
# Right table: one row per customer (cust_id is unique).
customers = pd.DataFrame({
    "cust_id": [1, 2, 3, 4, 5],
    "city":    ["NYC", "SF", "LA", "SEA", "BOS"],
})
# Left table: one row per order; some customers have several orders,
# and cust_id 6, 7 have NO matching customer.
orders = pd.DataFrame({
    "cust_id": [1, 1, 2, 3, 6, 6, 7],
    "amount":  [50, 20, 99, 15, 5, 8, 40],
})
print("orders rows   :", len(orders))      # 7
print("customers rows:", len(customers))   # 5

# === CONCATENATION: stack rows (same columns) ===
more_orders = pd.DataFrame({"cust_id": [2, 8], "amount": [12, 70]})
stacked = pd.concat([orders, more_orders], ignore_index=True)
print("after concat  :", len(stacked))     # 9  (7 + 2 stacked end-to-end)

# === JOINS / MERGES: line up rows on a key ===
for how in ["inner", "left", "right", "outer"]:
    m = pd.merge(orders, customers, on="cust_id", how=how)
    print(f"{how:>5} join rows:", len(m))
# inner join rows: 4   (only keys 1,1,2,3 match a customer)
# left  join rows: 7   (all orders; cust 6,6,7 get NaN city)
# right join rows: 6   (4 matched + customers 4,5 with NaN amount)
# outer join rows: 9   (4 matched + 3 order-only + 2 customer-only)

# === KEY MISMATCH -> silent zero matches (NO error raised) ===
bad = orders.copy()
bad["cust_id"] = bad["cust_id"].astype(str) + " "   # "1 ", "1 ", ...  string + space
oops = pd.merge(bad, customers, on="cust_id", how="left")
print("city all NaN? :", oops["city"].isna().all())  # True -- nothing matched!
# Fix: normalize BOTH keys before joining.
# bad["cust_id"] = bad["cust_id"].str.strip().astype(int)

# === validate=: assert the cardinality (raises if violated) ===
# We expect each customer (right) to map to MANY orders (left): a 1:m join.
m = pd.merge(orders, customers, on="cust_id", how="left", validate="1:m")
# If 'customers' had a duplicate cust_id, pandas would raise MergeError here.

# === indicator=True: audit which rows matched ===
audit = pd.merge(orders, customers, on="cust_id", how="outer", indicator=True)
print(audit["_merge"].value_counts().to_dict())
# {'both': 4, 'left_only': 3, 'right_only': 2}
#  both       = orders that found a customer
#  left_only  = orders with NO customer (cust 6,6,7)
#  right_only = customers with NO order  (cust 4,5)

## Visualize the data & results

_Joining two small tables (7 orders, 5 customers) on cust_id — how many rows does each join type return, and how do the matched / unmatched rows split?_

In [ ]:
import pandas as pd

customers = pd.DataFrame({"cust_id": [1, 2, 3, 4, 5],
                          "city": ["NYC", "SF", "LA", "SEA", "BOS"]})
orders = pd.DataFrame({"cust_id": [1, 1, 2, 3, 6, 6, 7],
                       "amount": [50, 20, 99, 15, 5, 8, 40]})

# Row count per join type.
for how in ["inner", "left", "right", "outer"]:
    print(how, len(pd.merge(orders, customers, on="cust_id", how=how)))
# inner 4 | left 7 | right 6 | outer 9

# Matched / unmatched split via indicator=True (on the outer join).
audit = pd.merge(orders, customers, on="cust_id", how="outer", indicator=True)
print(dict(audit["_merge"].value_counts()))
# {'both': 4, 'left_only': 3, 'right_only': 2}

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You left-join 50,000 orders to a customer table on cust_id. The result has 50,000 rows (good), but city is NaN for all of them. The join raised no error. What is the most likely cause and how do you confirm it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check the dtype of cust_id on both sides with .dtype. — _An int-vs-string mismatch (e.g. 3 vs "3") makes every key fail to match while the join still "succeeds"._
- Strip whitespace and normalize case/type: .astype(str).str.strip() on both keys. — _Hidden trailing spaces ("NYC ") or case ("nyc") are invisible but block matches._
- Re-run with indicator=True and inspect _merge.value_counts(). — _If everything is left_only, the keys are not matching at all &mdash; confirming the mismatch._

**Answer:** A key-type or whitespace/case mismatch: the keys silently fail to match, so the left join keeps all 50,000 rows but fills every right column with NaN, and raises no error. Confirm by comparing the two dtypes and by checking indicator=True &mdash; an all-left_only result is the giveaway. Fix by normalizing both keys (.astype(str).str.strip().str.lower()) before merging.

</details>

**Problem 2.** You merge a 1,000-row orders table to a 1,000-row promotions table on promo_code and the result has 7,400 rows. The count went UP after a join. What happened, and what one argument would have caught it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that an inner/left result larger than the left table means duplicated keys on both sides. — _If one promo_code maps to several promotions AND several orders, every pairing is emitted._
- Check uniqueness: orders["promo_code"].is_unique and promos["promo_code"].is_unique. — _A many-to-many join multiplies matching rows ($n\times m$ per key), exploding the row count._
- Add validate="m:1" (or "1:1") to the merge. — _pandas then raises a MergeError the moment the right side is not unique on the key, instead of silently exploding._

**Answer:** A many-to-many join (row explosion): promo_code is duplicated on both sides, so each key emits $n\times m$ rows and the total balloons past 1,000. Pass validate="m:1" (you expect each order to match exactly one promotion) and pandas raises a MergeError instead of quietly multiplying rows. Always compare row counts before and after a join.

</details>

**Problem 3.** Decide for each task: concatenate or merge? (a) You have 12 monthly CSVs with identical columns and want one full-year table. (b) You have an orders table and a customers table and want each order labeled with the customer's city.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Ask whether the columns are the same (stack rows) or you are pulling in new columns via a shared key. — _Same schema, more rows is concatenation; same rows, more columns by key is a merge._
- For (a), the 12 files share columns and you want them end-to-end. — _pd.concat([jan, feb, ...]) stacks rows; no key matching is needed._
- For (b), you attach the city column by matching cust_id. — _pd.merge(orders, customers, on="cust_id", how="left") looks up each order's customer._

**Answer:** (a) Concatenate: identical columns, stacking rows &mdash; pd.concat([jan, feb, ...], ignore_index=True). (b) Merge: pulling a new column in by a shared key &mdash; pd.merge(orders, customers, on="cust_id", how="left"). Same columns, more rows &rarr; concat; same rows, more columns by key &rarr; merge.

</details>